# Phase B - CellChat / LIANA (communication features)

Communication-Aware ML for MDD. This notebook runs on Colab (CPU, High-RAM)
and reads the Phase A checkpoint from Drive.

**Step 0** below is a read-only diagnostic: per-donor x cell-type nucleus
counts, used to decide granularity (broad vs hybrid) *before* we build any
features.

### Dependencies

In [ ]:
# scanpy is not preinstalled on Colab. anndata + h5py come with it.
# pyarrow is for caching obs as parquet.
!pip install -q scanpy pyarrow

### Boot cell

Mounts Drive, pulls the latest repo (so `src/functions.py` is current), and
puts `src/` on the path. `%autoreload 2` means no kernel restart after a pull.

In [ ]:
import os, sys
from google.colab import drive
drive.mount('/content/drive')

PROJECT_ROOT   = '/content/drive/MyDrive/MLCB_team_project'
RAW_DIR        = os.path.join(PROJECT_ROOT, 'data', 'raw')
CHECKPOINT_DIR = os.path.join(PROJECT_ROOT, 'data', 'checkpoints')
for d in (RAW_DIR, CHECKPOINT_DIR):
    os.makedirs(d, exist_ok=True)

REPO_URL = 'https://github.com/andreas824/MLCB_team_project.git'
REPO_DIR = '/content/MLCB_team_project'
if not os.path.exists(REPO_DIR):
    !git clone {REPO_URL} {REPO_DIR}
else:
    !cd {REPO_DIR} && git pull

sys.path.insert(0, os.path.join(REPO_DIR, 'src'))

# autoreload is optional: Colab's Python 3.12 + IPython 7.34 ship a broken
# autoreload (uses the removed `imp` module), so never let it break boot.
try:
    _ip = get_ipython()
    _ip.run_line_magic('load_ext', 'autoreload')
    _ip.run_line_magic('autoreload', '2')
except Exception as e:
    print(f'autoreload unavailable ({e}); continuing. Restart runtime after a pull if functions.py changes.')

from functions import build_condition_map, load_dataset  # noqa: F401

### Step 0 - per-donor x cell-type count diagnostic

Read-only sanity check. Decides granularity (broad vs hybrid) before we build
any features. We only need `.obs`, so we open the checkpoint in backed mode and
never load the 5.4 GB expression matrix into RAM.

In [ ]:
# === Phase B - Step 0: per-donor x cell-type count diagnostic ===
import os
import numpy as np
import pandas as pd
import scanpy as sc

# CHECKPOINT_DIR comes from the boot cell; fall back to the standard path.
CHECKPOINT_DIR = globals().get(
    'CHECKPOINT_DIR',
    '/content/drive/MyDrive/MLCB_team_project/data/checkpoints',
)
CKPT = os.path.join(CHECKPOINT_DIR, 'phaseA_final_for_cellchat.h5ad')

adata = sc.read_h5ad(CKPT, backed='r')   # backed='r' => obs only, no matrix in RAM
obs = adata.obs.copy()

# Optional: cache obs as a small parquet so future diagnostics skip the big file.
obs.to_parquet(os.path.join(CHECKPOINT_DIR, 'phaseA_obs.parquet'))

# ---- 0. dataset sanity: 71 donors, sex x condition layout -------------------
donor_meta = obs.drop_duplicates('donor_id').set_index('donor_id')[['sex', 'condition']]
print(f"donors: {donor_meta.shape[0]}   nuclei: {obs.shape[0]:,}")
print("\nsex x condition (donors):")
print(pd.crosstab(donor_meta['sex'], donor_meta['condition'], margins=True))

# ---- helpers ----------------------------------------------------------------
THRESHOLDS = [5, 10, 20, 30, 50]   # candidate min-cells-per-type-per-donor floors
N_DONORS = donor_meta.shape[0]

def per_donor_table(col):
    # donor (rows) x cell-type (cols) nucleus-count matrix.
    return pd.crosstab(obs['donor_id'], obs[col])

def summarize(counts, label):
    print(f"\n{'='*70}\n{label}: per-donor count distribution\n{'='*70}")
    desc = counts.describe(percentiles=[.1, .25, .5, .75, .9]).T
    desc = desc[['min', '10%', '50%', '90%', 'max']].round(0).astype(int)
    desc.insert(0, 'donors_with_zero', (counts == 0).sum().values)
    print(desc.to_string())
    print(f"\n# donors (of {N_DONORS}) with >= threshold cells of each type:")
    surv = pd.DataFrame({f">= {t}": (counts >= t).sum() for t in THRESHOLDS})
    print(surv.to_string())

# ---- 1. BROAD granularity ---------------------------------------------------
broad = per_donor_table('cell_type_broad')
if 'Mix' in broad.columns:               # the authors' mixed-profile cluster
    broad = broad.drop(columns='Mix')
summarize(broad, "BROAD cell types (cell_type_broad)")

# ---- 2. FINE clusters the hypothesis actually names -------------------------
fine = per_donor_table('cell_type_fine')
PAIR = ['Mic1', 'ExN10_L46']
missing = [c for c in PAIR if c not in fine.columns]
if missing:
    print(f"\n!! WARNING: expected fine clusters not found: {missing}")
PAIR = [c for c in PAIR if c in fine.columns]
summarize(fine[PAIR], "HYPOTHESIS-PAIR fine clusters")

# ---- 3. fine cluster as a fraction of its broad parent ----------------------
parent = {'Mic1': 'Mic', 'ExN10_L46': 'ExN'}
print(f"\n{'='*70}\nfine-vs-parent share (median per-donor % of parent broad type)\n{'='*70}")
for fine_cl in PAIR:
    p = parent[fine_cl]
    if p in broad.columns:
        frac = (fine[fine_cl] / broad[p].replace(0, np.nan) * 100)
        print(f"  {fine_cl:11s} as % of {p:3s}: "
              f"median {frac.median():.1f}%  (range {frac.min():.1f}-{frac.max():.1f}%)")

# ---- 4. decision helper -----------------------------------------------------
print(f"\n{'='*70}\nGRANULARITY VERDICT HELPER\n{'='*70}")
for fine_cl in PAIR:
    n10 = int((fine[fine_cl] >= 10).sum())
    n20 = int((fine[fine_cl] >= 20).sum())
    print(f"  {fine_cl:11s}: {n10}/{N_DONORS} donors >=10 cells, "
          f"{n20}/{N_DONORS} >=20 cells")
print("\nRule of thumb: if BOTH clusters clear ~10-20 cells in MOST donors -> HYBRID")
print("is feasible (split Mic1/ExN10_L46 out). If many donors fall short -> stay BROAD.")